In [ ]:
import json
import os
from pathlib import Path
from typing import Literal

import pandas as pd
from anthropic import Anthropic
from anthropic.types.message_create_params import (
    MessageCreateParamsNonStreaming,
)
from anthropic.types.messages.batch_create_params import (
    Request,
)
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

## Settings

In [ ]:
# Read .env file
load_dotenv(
    dotenv_path=Path().resolve().parent.parent / ".env",
    override=False,
)

# Set OpenAI API key
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

# Set Anthropic API key
ANTHROPIC_API_KEY = os.environ["ANTHROPIC_API_KEY"]

In [ ]:
# Set directories
DATASET_DIR = Path("./dataset_processed")
SYSTEM_PROMPT_DIR = Path("./system_prompt")
USER_PROMPT_DIR = Path("./user_prompt")

# Create directory for batch jsonl files
TIME_TAG = pd.Timestamp.now().strftime("%Y%m%d%H%M%S")
BATCH_INPUT_DIR = Path("./batch_input") / f"batch_input_{TIME_TAG}"
BATCH_INPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Model lists
OPENAI_NON_REASONING_MODELS = [
    "gpt-4.1-mini-2025-04-14",
    "gpt-4o-mini-2024-07-18",
]
OPENAI_REASONING_MODELS = [
    "gpt-5-mini-2025-08-07",
    "o4-mini-2025-04-16",
]
OPENAI_MODELS = OPENAI_NON_REASONING_MODELS + OPENAI_REASONING_MODELS

ANTHROPIC_MODELS = [
    "claude-sonnet-4-5-20250929",
    "claude-haiku-4-5-20251001",
    "claude-3-5-haiku-20241022",
]

In [ ]:
# Settings
DATASET_NAME_LIST = [
    "BH_1",
    "DA",
    "p3ht",
    "suzuki",
]  # ["BH_1", "DA", "alkox", "oer_plate_a", "p3ht", "photo_pce10", "photo_wf3", "suzuki_edbo", "suzuki"]
MODEL_LIST = ["gpt-5-mini-2025-08-07"]
TEST_MODE = False  # True or False

COMMON_MAX_OUTPUT_TOKENS = 8192
COMMON_REASONING_EFFORT = "medium"  # "minimal", "low", "medium", "high"
COMMON_TEMPERATURE = 0.5

REPEAT_NUM = 5

## Functions for batch processing

In [ ]:
# Response format model
class ResponseFormat(BaseModel):
    answer_token: Literal["A", "B", "C"] = Field(description="answer token: 'A', 'B', or 'C'")

In [ ]:
# Function to generate OpenAI batch input file
def generate_openai_batch_input_file(
    dataset_name,
    model,
    max_output_tokens,
    reasoning_effort,
    temperature,
    test_mode=False,
    reverse_mode=False,
):
    # Read data
    dataset = pd.read_csv(DATASET_DIR / f"dataset_{dataset_name}_pair_wo_obj.csv").astype({"ID_A": int, "ID_B": int})
    if test_mode:
        dataset = dataset.iloc[0:3, :]

    if reverse_mode:
        new_columns = []
        for col in dataset.columns:
            if col.endswith("_A"):
                new_columns.append(col[:-2] + "_B")
            elif col.endswith("_B"):
                new_columns.append(col[:-2] + "_A")
            else:
                new_columns.append(col)
        dataset.columns = new_columns

    print(f"{len(dataset)} rows")

    # Read system prompt
    with open(SYSTEM_PROMPT_DIR / f"dataset_{dataset_name}.txt", "r", encoding="utf-8") as f:
        system_content = f.read()

    # Read user prompt template
    with open(USER_PROMPT_DIR / f"dataset_{dataset_name}.txt", "r", encoding="utf-8") as f:
        user_content_template = f.read()

    if not reverse_mode:
        batch_input_filepath = f"batch_input_{dataset_name}_{model}.jsonl"
    else:
        batch_input_filepath = f"batch_input_{dataset_name}_{model}_reverse.jsonl"

    with open(BATCH_INPUT_DIR / batch_input_filepath, "w", encoding="utf-8") as f:
        for i, dataset_row in dataset.iterrows():
            dataset_row_dict = dataset_row.to_dict()
            dataset_row_dict = {k: v for k, v in dataset_row_dict.items() if (k != "ID_A") and (k != "ID_B")}

            # Generate user prompt
            user_content = user_content_template.format(**dataset_row_dict)

            # Generate response format
            schema = ResponseFormat.model_json_schema()
            schema["additionalProperties"] = False

            # Generate request
            request = {
                "custom_id": f"request-A{int(dataset_row['ID_A'])}-B{int(dataset_row['ID_B'])}",
                "method": "POST",
                "url": "/v1/responses",
                "body": {
                    "model": model,
                    "input": [
                        {"role": "system", "content": system_content},
                        {"role": "user", "content": user_content},
                    ],
                    "text": {
                        "format": {
                            "type": "json_schema",
                            "name": "response",
                            "strict": True,
                            "schema": schema,
                        },
                    },
                },
            }
            if max_output_tokens is not None:
                request["body"]["max_output_tokens"] = max_output_tokens
            if reasoning_effort is not None:
                request["body"]["reasoning"] = {"effort": reasoning_effort}
            if temperature is not None:
                request["body"]["temperature"] = temperature

            f.write(json.dumps(request, ensure_ascii=False) + "\n")

In [ ]:
# Function to generate Anthropic batch input file
def generate_anthropic_requests(
    dataset_name,
    model,
    max_output_tokens,
    temperature,
    test_mode=False,
    reverse_mode=False,
):
    # Read data
    dataset = pd.read_csv(DATASET_DIR / f"dataset_{dataset_name}_pair_wo_obj.csv").astype({"ID_A": int, "ID_B": int})
    if test_mode:
        dataset = dataset.iloc[0:3, :]

    if reverse_mode:
        new_columns = []
        for col in dataset.columns:
            if col.endswith("_A"):
                new_columns.append(col[:-2] + "_B")
            elif col.endswith("_B"):
                new_columns.append(col[:-2] + "_A")
            else:
                new_columns.append(col)
        dataset.columns = new_columns

    print(f"{len(dataset)} rows")

    # Read system prompt
    with open(SYSTEM_PROMPT_DIR / f"dataset_{dataset_name}.txt", "r", encoding="utf-8") as f:
        system_content = f.read()

    # Read user prompt template
    with open(USER_PROMPT_DIR / f"dataset_{dataset_name}.txt", "r", encoding="utf-8") as f:
        user_content_template = f.read()

    requests = []
    for i, dataset_row in dataset.iterrows():
        dataset_row_dict = dataset_row.to_dict()
        dataset_row_dict = {k: v for k, v in dataset_row_dict.items() if (k != "ID_A") and (k != "ID_B")}

        # Generate user prompt
        user_content = user_content_template.format(**dataset_row_dict)

        # Generate response format
        schema = ResponseFormat.model_json_schema()
        tool_def = {
            "name": "JSON_schema_response_format",
            "description": "JSON Schema as the response format.",
            "input_schema": schema,
        }

        # Generate request
        request = Request(
            custom_id=f"request-A{int(dataset_row['ID_A'])}-B{int(dataset_row['ID_B'])}",
            params=MessageCreateParamsNonStreaming(
                model=model,
                max_tokens=max_output_tokens,
                temperature=temperature,
                system=system_content,
                messages=[
                    {
                        "role": "user",
                        "content": user_content,
                    }
                ],
                tools=[tool_def],
                tool_choice={"type": "tool", "name": "JSON_schema_response_format"},
            ),
        )

        requests.append(request)

    # Save to jsonl file
    if not reverse_mode:
        batch_input_filepath = f"batch_input_{dataset_name}_{model}.jsonl"
    else:
        batch_input_filepath = f"batch_input_{dataset_name}_{model}_reverse.jsonl"
    with open(BATCH_INPUT_DIR / batch_input_filepath, "w", encoding="utf-8") as f:
        for request in requests:
            f.write(json.dumps(request, ensure_ascii=False) + "\n")

    return requests

## Batch processing

In [ ]:
# Batch processing
logs = []
for dataset_name in DATASET_NAME_LIST:
    for model in MODEL_LIST:
        print("-" * 100)
        print(f"dataset: {dataset_name:<15}| model: {model:<15}")

        for i in range(REPEAT_NUM):
            for reverse_mode in [False, True]:
                print(f"repeat: {i + 1}, reverse mode: {reverse_mode}")

                if model in OPENAI_MODELS:
                    # Generate batch input file
                    if model in OPENAI_NON_REASONING_MODELS:
                        reasoning_effort = None
                        temperature = COMMON_TEMPERATURE
                    elif model in OPENAI_REASONING_MODELS:
                        reasoning_effort = COMMON_REASONING_EFFORT
                        temperature = None
                    else:
                        raise ValueError(f"Unsupported model: {model}")

                    generate_openai_batch_input_file(
                        dataset_name=dataset_name,
                        model=model,
                        max_output_tokens=COMMON_MAX_OUTPUT_TOKENS,
                        reasoning_effort=reasoning_effort,
                        temperature=temperature,
                        test_mode=TEST_MODE,
                        reverse_mode=reverse_mode,
                    )

                    # Read batch input file
                    if not reverse_mode:
                        batch_input_filepath = BATCH_INPUT_DIR / f"batch_input_{dataset_name}_{model}.jsonl"
                    else:
                        batch_input_filepath = BATCH_INPUT_DIR / f"batch_input_{dataset_name}_{model}_reverse.jsonl"
                    with open(batch_input_filepath, "rb") as f:
                        lines = f.readlines()
                        n_samples_input = len(lines)
                    print(f"{n_samples_input} rows")

                    # Batch processing
                    client_openai = OpenAI(api_key=OPENAI_API_KEY)
                    batch_input_file = client_openai.files.create(
                        file=open(batch_input_filepath, "rb"), purpose="batch"
                    )
                    batch_input_file_id = batch_input_file.id
                    batch = client_openai.batches.create(
                        input_file_id=batch_input_file.id,
                        endpoint="/v1/responses",
                        completion_window="24h",
                    )
                    batch_id = batch.id
                    print(f"Batch ID: {batch_id}")

                elif model in ANTHROPIC_MODELS:
                    # Generate batch requests
                    if model in ANTHROPIC_MODELS:
                        reasoning_effort = None
                        temperature = COMMON_TEMPERATURE
                    else:
                        raise ValueError(f"Unsupported model: {model}")

                    requests = generate_anthropic_requests(
                        dataset_name=dataset_name,
                        model=model,
                        max_output_tokens=COMMON_MAX_OUTPUT_TOKENS,
                        temperature=temperature,
                        test_mode=TEST_MODE,
                        reverse_mode=reverse_mode,
                    )
                    n_samples_input = len(requests)
                    print(f"{n_samples_input} rows")

                    # Batch processing
                    client_anthropic = Anthropic(api_key=ANTHROPIC_API_KEY)
                    batch_input_file_id = ""
                    batch = client_anthropic.messages.batches.create(requests=requests)
                    batch_id = batch.id
                    print(f"Batch ID: {batch_id}")

                else:
                    raise ValueError(f"Unsupported model: {model}")

                logs.append(
                    {
                        "dataset_name": dataset_name,
                        "model": model,
                        "max_output_tokens": COMMON_MAX_OUTPUT_TOKENS,
                        "reasoning_effort": reasoning_effort,
                        "temperature": temperature,
                        "n_samples_input": n_samples_input,
                        "repeat": i + 1,
                        "forward_or_reverse": "forward" if not reverse_mode else "reverse",
                        "batch_input_file_id": batch_input_file_id,
                        "batch_id": batch_id,
                    }
                )

# Save logs
logs_df = pd.DataFrame(logs)
logs_df.to_csv(BATCH_INPUT_DIR / "batch_input_logs.csv", index=False)